# 📑 Модуль 10. Урок 2: Разметка неструктурированных отзывов и тикетов саппорта

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_10_llm_api/lesson_02_parsing_unstructured_data.ipynb)

---

## 1. Реальный кейс: Автоматический аудит 800 жалоб клиентов e-commerce

Работаем с реальным датасетом отзывов `customer_reviews.csv`.
Извлекаем темы жалоб и тональность без ручного чтения.


In [ ]:
import pandas as pd

reviews = pd.read_csv(r"d:\vibe_app\main_data_analytics\course\datasets\raw\customer_reviews.csv")

# Функция симуляции работы LLM по извлечению сущностей и тематик
def analyze_review_with_llm(text: str) -> dict:
    t = text.lower()
    topics = []
    if any(k in t for k in ['курьер', 'доставка', 'опоздал', 'доставили']):
        topics.append('Логистика и доставка')
    if any(k in t for k in ['списали', 'деньги', 'вернуть', 'цена', 'подписку']):
        topics.append('Оплата и биллинг')
    if any(k in t for k in ['баги', 'крашится', 'тормозит', 'crashes', 'не работает']):
        topics.append('Баги приложения')
    if any(k in t for k in ['поддержка', 'чат', 'саппорт', 'service']):
        topics.append('Качество саппорта')
        
    return {
        'topics': topics if topics else ['Прочее'],
        'has_critical_issue': int(any(k in t for k in ['верните', 'претензию', 'списали', 'crashes']))
    }

# Применяем анализ к базе
results = reviews['text'].apply(analyze_review_with_llm)
reviews['topics'] = [r['topics'] for r in results]
reviews['is_critical'] = [r['has_critical_issue'] for r in results]

# Сводный отчет для операционного директора
exploded = reviews.explode('topics')
summary = exploded.groupby('topics').agg(
    total_reviews=('review_id', 'count'),
    critical_cases=('is_critical', 'sum')
).sort_values(by='critical_cases', ascending=False)

print("=== РЕЗУЛЬТАТ АВТОМАТИЧЕСКОЙ КЛАССИФИКАЦИИ ЖАЛОБ ===")
print(summary)
